# Rule Engine Demo — German Credit Dataset

This notebook applies the three-valued logic rule engine from `engine.py` to the
classic **German Credit** dataset (1 000 applicants, 20 features, target: `good`/`bad`).

### Four credit-decision rules

| # | Name | Logic |
|---|------|-------|
| 1 | **premium_approval** | Healthy checking AND good credit history AND short loan |
| 2 | **safe_profile** | Age ≥ 25 AND (good savings OR stable employment) AND moderate loan |
| 3 | **young_large_loan** | Age < 30 AND large loan (risk flag) |
| 4 | **auto_reject** | Negative checking AND poor credit history |

Leaf conditions are boolean values pre-computed from raw features and passed as the
`data` dict.  Missing a feature → its leaves become `"unknown"`, propagating through
Strong Kleene logic.

## 1  Imports & data loading

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from engine import evaluate, provenance, leaf, and_, or_, not_

# Fetch dataset (cached after first download)
dataset = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')
df = dataset.frame.copy()

print(f"Shape: {df.shape}")
df.head(3)

## 2  Boolean condition extraction

Each leaf name maps to a single boolean expression.  Converting upfront keeps the
rules readable and the engine's `data` dict flat.

In [ ]:
def extract_conditions(row: pd.Series) -> dict:
    """Return a flat {leaf_name: bool} dict for one applicant row."""
    return {
        # --- age ---
        "age_ge_25":          row["age"] >= 25,
        "young":              row["age"] < 30,

        # --- credit amount ---
        "large_loan":         row["credit_amount"] > 5_000,
        "moderate_loan":      row["credit_amount"] <= 7_500,

        # --- loan duration ---
        "short_duration":     row["duration"] <= 24,

        # --- checking account ---
        "negative_checking":  row["checking_status"] == "<0",
        "healthy_checking":   row["checking_status"] == ">=200",

        # --- savings ---
        "good_savings":       row["savings_status"] in ("500<=X<1000", ">=1000"),

        # --- employment stability ---
        "stable_employment":  row["employment"] in ("4<=X<7", ">=7"),

        # --- credit history ---
        "good_credit_hist":   row["credit_history"] in ("all paid", "no credits/all paid"),
        "poor_credit_hist":   row["credit_history"] in ("delayed previously",
                                                          "critical/other existing credit"),
    }

# Quick sanity check on first row
extract_conditions(df.iloc[0])

## 3  Rule definitions

In [ ]:
# Rule 1: healthy account + clean history + not a multi-year commitment
rule_premium = and_(
    leaf("healthy_checking"),
    leaf("good_credit_hist"),
    leaf("short_duration"),
)

# Rule 2: mature borrower, some financial cushion, loan not excessive
rule_safe = and_(
    leaf("age_ge_25"),
    or_(leaf("good_savings"), leaf("stable_employment")),
    leaf("moderate_loan"),
)

# Rule 3: risk flag — young applicant with a large loan
rule_young_large = and_(
    leaf("young"),
    leaf("large_loan"),
)

# Rule 4: automatic rejection signals
rule_reject = and_(
    leaf("negative_checking"),
    leaf("poor_credit_hist"),
)

RULES = {
    "premium_approval": rule_premium,
    "safe_profile":     rule_safe,
    "young_large_loan": rule_young_large,
    "auto_reject":      rule_reject,
}

print("Rules defined:", list(RULES))

## 4  Evaluate all applicants

In [ ]:
def run_rules(df_in: pd.DataFrame) -> pd.DataFrame:
    records = []
    for idx, row in df_in.iterrows():
        data = extract_conditions(row)
        rec = {"applicant": idx, "actual": row["class"]}
        for name, rule in RULES.items():
            rec[name] = evaluate(rule, data)
        records.append(rec)
    return pd.DataFrame(records).set_index("applicant")

results = run_rules(df)
results.head(10)

## 5  Outcome summary

In [ ]:
summary_rows = []
for rule_name in RULES:
    counts = results[rule_name].value_counts()
    summary_rows.append({
        "rule":    rule_name,
        "True":    int(counts.get(True,      0)),
        "False":   int(counts.get(False,     0)),
        "unknown": int(counts.get("unknown", 0)),
    })

summary = pd.DataFrame(summary_rows).set_index("rule")
summary["total"] = summary.sum(axis=1)
print(summary)

## 6  Provenance — which conditions determined the outcome?

We sample a few applicants from each rule's True/False bucket and display the
leaf conditions that actually drove the decision.

In [ ]:
def show_provenance(rule_name: str, outcome, n: int = 5):
    rule = RULES[rule_name]
    matched = results[results[rule_name] == outcome].head(n)
    print(f"\n{'='*60}")
    print(f"Rule: {rule_name!r}  |  outcome={outcome!r}  |  showing {len(matched)} applicants")
    print(f"{'='*60}")
    for idx in matched.index:
        row  = df.loc[idx]
        data = extract_conditions(row)
        prov = provenance(rule, data)
        actual = row["class"]
        print(f"  applicant {idx:>4d}  actual={actual:<5}  determining leaves: {prov}")

show_provenance("premium_approval", True)
show_provenance("premium_approval", False,  n=4)
show_provenance("safe_profile",     True)
show_provenance("young_large_loan", True)
show_provenance("auto_reject",      True)

## 7  Partial-data scenario

Suppose downstream pipeline drops the **`age`** column (e.g. a privacy-preserving
transform, a schema change, or a late-arriving feed).  Any leaf that depended on age
(`age_ge_25`, `young`) disappears from the data dict — `evaluate()` returns
`"unknown"` for those leaves instead of crashing.

Strong Kleene propagates the unknown **unless the remaining leaves already determine
the outcome** (e.g. `safe_profile` can still be `False` if `moderate_loan` is False,
even with age unknown).

In [ ]:
def extract_conditions_no_age(row: pd.Series) -> dict:
    """Same as extract_conditions but omits age-derived leaves."""
    d = extract_conditions(row)
    d.pop("age_ge_25")
    d.pop("young")
    return d

def run_rules_no_age(df_in: pd.DataFrame) -> pd.DataFrame:
    records = []
    for idx, row in df_in.iterrows():
        data = extract_conditions_no_age(row)
        rec = {"applicant": idx, "actual": row["class"]}
        for name, rule in RULES.items():
            rec[name] = evaluate(rule, data)
        records.append(rec)
    return pd.DataFrame(records).set_index("applicant")

results_no_age = run_rules_no_age(df)
results_no_age.head(10)

In [ ]:
# Compare outcome distributions: full data vs. age-dropped
comparison_rows = []
for rule_name in RULES:
    for label, res in [("full", results), ("no_age", results_no_age)]:
        counts = res[rule_name].value_counts()
        comparison_rows.append({
            "rule":    rule_name,
            "data":    label,
            "True":    int(counts.get(True,      0)),
            "False":   int(counts.get(False,     0)),
            "unknown": int(counts.get("unknown", 0)),
        })

comp = (pd.DataFrame(comparison_rows)
          .set_index(["rule", "data"]))
print(comp.to_string())

In [ ]:
# Deep-dive: applicants whose safe_profile verdict CHANGED when age was removed
changed = results["safe_profile"] != results_no_age["safe_profile"]
changed_idx = changed[changed].index
print(f"{len(changed_idx)} applicants changed verdict for 'safe_profile' when age was removed.")

rule = RULES["safe_profile"]
print("\nSample — full data vs. no-age provenance:")
for idx in changed_idx[:6]:
    row         = df.loc[idx]
    data_full   = extract_conditions(row)
    data_no_age = extract_conditions_no_age(row)

    v_full   = evaluate(rule, data_full)
    v_no_age = evaluate(rule, data_no_age)
    p_full   = provenance(rule, data_full)
    p_no_age = provenance(rule, data_no_age)

    print(f"  applicant {idx:>4d}  age={row['age']:>3}  "
          f"full={str(v_full):<8} prov={p_full}")
    print(f"  {' ':>15}  no_age: {str(v_no_age):<8} prov={p_no_age}")
    print()

In [ ]:
# Rules unaffected by age: premium_approval and auto_reject should be identical
for rule_name in ["premium_approval", "auto_reject"]:
    same = (results[rule_name] == results_no_age[rule_name]).all()
    print(f"{rule_name}: results identical when age removed? {same}")

## 8  Key takeaways

- **Strong Kleene `unknown` propagates conservatively**: a missing leaf only forces an
  `unknown` result when it would actually change the outcome — already-determined
  branches (e.g. a False short-circuit in AND) still resolve correctly.
- **`provenance()` pinpoints root causes**: for approved applicants it names the
  supporting evidence; for rejected ones it names the blocking conditions; for
  `unknown` results it names the missing data that prevented a decision.
- **Rules that don't reference age** (`premium_approval`, `auto_reject`) are
  completely unaffected by the missing column, demonstrating graceful degradation.